# M2 정규화 진단 — L2 완화 병렬 실험

이 노트북은 기존 학습예산 진단과 **별도 Colab 런타임에서 병렬 실행**합니다. `M1`과 `M2` 모두 L2 정규화를 `1e-3`에서 `1e-4`로 낮추고 300 epoch까지 학습하며, 25 epoch마다 개발분할을 평가합니다.

> 기존 M2가 M1보다 낮았던 이유가 N/V 표현을 과도하게 억제한 정규화였는가?

고정 조건은 ID 64차원, N/V 축별 4차원, `rho=0.05`, 2층, K=1 균일 음성입니다. **L2 외에는 현재 baseline 학습예산 실험과 같습니다.** M1도 동일한 L2로 다시 학습하므로 M2만 유리하게 바꾸지 않습니다.

M2는 사용자의 `q_N`·`q_V`가 개인 구매이력–후보상품 적합 블록을 조절하며, 같은 forward·BPR 손실·optimizer 안에서 학습됩니다. 외부 보정이나 재정렬은 없습니다.

**범위 주의:** `q_C=percentile(N×V)`는 사용하지 않으므로 전체 historical CLV 수준 모형이 아니라 **CLV 구성요소 기반 M2**로 판독합니다.

결과는 baseline 실험과 다른 Drive 폴더에 저장해 병렬 실행 중 체크포인트·JSON 충돌을 막습니다. 이번 단일 seed에서는 조건이나 epoch를 선택하지 않고 학습곡선과 matched `M2−M1`만 보고합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '71ea2bd683ad6ba5387856764533b0cd9587eb00'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m2_capacity_search as search

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
LIGHT_L2_OUT_DIR = '/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_m2_capacity_search_light_l2_parallel_v1'
cfg = search.configure_capacity_search(
    conditions=('light_l2',),
    out_dir=LIGHT_L2_OUT_DIR,
)
summary = search.preflight_summary(cfg)
specs = search.arm_specifications(cfg)
assert list(summary['conditions']) == ['light_l2']
assert summary['conditions']['light_l2']['pref_reg'] == 1e-4
assert [spec['model_id'] for spec in specs] == [search.M1_MODEL_ID, search.M2_MODEL_ID]
assert all(spec['pref_reg'] == 1e-4 for spec in specs)
assert summary['protocol_epoch'] in summary['evaluated_at_epochs']
assert len(specs) == 2
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
curve = search.run_capacity_search(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy(); view.attrs = {}; display(view)

print('1) 조건별 학습곡선 (25 epoch 간격, clv_score_share = CLV 블록이 점수에서 차지하는 비중)')
show(curve)
print('2) 같은 조건·같은 시드에서 M2 - M1')
show(curve.attrs['gap'])
print('3) 판독')
print(json.dumps(curve.attrs['reading'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(curve.attrs['result_paths'], ensure_ascii=False, indent=2))


In [ ]:
# 교수님 보고용 그림: L2 완화 상태에서 100 epoch 이후 M2-M1 격차가 회복되는가
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for model_id, line in curve.groupby('model_id'):
    axes[0].plot(line.epoch, line['recall@10'], marker='o', markersize=3,
                 label=model_id.split('_')[0].upper())
gap = curve.attrs['gap']
axes[1].plot(gap.epoch, gap['recall@10'], marker='o', markersize=3,
             label='light_l2')
for ax, title in zip(axes, ['L2=1e-4 개발 Recall@10', 'L2=1e-4 M2 - M1 (Recall@10)']):
    ax.axvline(100, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel('epoch'); ax.set_title(title); ax.legend(fontsize=8)
axes[1].axhline(0, color='black', linewidth=0.8)
plt.tight_layout(); plt.show()
